# Summary

# API

In [1]:
from transformers import AutoTokenizer

# Load.
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer_bck = tokenizer.backend_tokenizer

# Structure.
print(f"[Normalizer] {tokenizer_bck.normalizer}")
print(f"[Pre-tokenizer] {tokenizer_bck.pre_tokenizer}")
print(f"[Model] {tokenizer_bck.model}")
print(f"[Post-processor] {tokenizer_bck.post_processor}", '\n')

# Text.
text = [
    "Hello, transformer!",
    "What is your name?",
]
print(f"[Text] {text}")

# Encode: text -> id.
encode = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=10,
    return_tensors="pt",
)
print(f"[Encode] {encode['input_ids']}")

# Decode: id -> text.
print(f"[Decode] {tokenizer.decode(encode['input_ids'])}")

# token id -> token.
tokens = tokenizer.convert_ids_to_tokens(encode['input_ids'][0])
print(f"[id -> token] {tokens}")

# token -> id.
ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"[token -> id] {ids}")

# Methods and attributes.
tokenizer.is_fast               # whether loaded as a fast tokenizer.

tokenizer.all_special_tokens    # ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
tokenizer.all_special_ids       # [100, 102, 0, 101, 103]
tokenizer.pad_token
tokenizer.unk_token
tokenizer.bos_token
tokenizer.eos_token

tokenizer.padding_side          # left / right.
tokenizer.truncation_side       # left / right.

tokenizer.get_vocab()           # get vocab.
tokenizer.vocab_size            # vocab size.

tokenizer.model_max_length      # maximum n_tokens of the transformer.

c:\Users\yana\Desktop\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Normalizer] BertNormalizer(clean_text=True, handle_chinese_chars=True, strip_accents=None, lowercase=True)
[Pre-tokenizer] BertPreTokenizer()
[Model] WordPiece(unk_token="[UNK]", continuing_subword_prefix="##", max_input_chars_per_word=100, vocab={"[PAD]":0, "[unused0]":1, "[unused1]":2, "[unused2]":3, "[unused3]":4, ...})
[Post-processor] TemplateProcessing(single=[SpecialToken(id="[CLS]", type_id=0), Sequence(id=A, type_id=0), SpecialToken(id="[SEP]", type_id=0)], pair=[SpecialToken(id="[CLS]", type_id=0), Sequence(id=A, type_id=0), SpecialToken(id="[SEP]", type_id=0), Sequence(id=B, type_id=1), SpecialToken(id="[SEP]", type_id=1)], special_tokens={"[CLS]":SpecialToken(id="[CLS]", ids=[101], tokens=["[CLS]"]), "[SEP]":SpecialToken(id="[SEP]", ids=[102], tokens=["[SEP]"])}) 

[Text] ['Hello, transformer!', 'What is your name?']
[Encode] tensor([[  101,  7592,  1010, 10938,  2121,   999,   102,     0,     0,     0],
        [  101,  2054,  2003,  2115,  2171,  1029,   102,     0,     

512

# 1. Normalizers

In [2]:
from tokenizers.normalizers import (
    Lowercase, 
    NFD,
    StripAccents,
    Sequence,
    Strip,
    BertNormalizer,
)

# Lowercase.
print('Lowercase: ', Lowercase().normalize_str(text[0]))

# Strip accents.
norm = Sequence([NFD(), StripAccents()])
print(f"Accent stripping: café -> {norm.normalize_str("café")}")

# Strip whitespaces.
txt = "   hello   "
norm = Strip(left=True, right=True)
print(f"Strip whitespaces: {txt} -> {norm.normalize_str(txt)}")

# BertNormalizer.
norm = BertNormalizer()
print(f"BertNormalizer: {text[0]} -> {norm.normalize_str(text[0])}")

Lowercase:  hello, transformer!
Accent stripping: café -> cafe
Strip whitespaces:    hello    -> hello
BertNormalizer: Hello, transformer! -> hello, transformer!


# 2. Pre-tokenizers

In [3]:
from tokenizers import pre_tokenizers

text = "I love machine learning!"

# Whitespace.
pre = pre_tokenizers.Whitespace()
print("Whitespace: ", pre.pre_tokenize_str(text))

# BERT.
pre = pre_tokenizers.BertPreTokenizer()
print("BERT: ", pre.pre_tokenize_str(text))

# Byte-level (gpt-2 or RoBERTa).
pre = pre_tokenizers.ByteLevel()
print("Byte-level: ", pre.pre_tokenize_str(text))

# SentencePiece.
pre = pre_tokenizers.Metaspace()
print("SentencePiece: ", pre.pre_tokenize_str(text))

Whitespace:  [('I', (0, 1)), ('love', (2, 6)), ('machine', (7, 14)), ('learning', (15, 23)), ('!', (23, 24))]
BERT:  [('I', (0, 1)), ('love', (2, 6)), ('machine', (7, 14)), ('learning', (15, 23)), ('!', (23, 24))]
Byte-level:  [('ĠI', (0, 1)), ('Ġlove', (1, 6)), ('Ġmachine', (6, 14)), ('Ġlearning', (14, 23)), ('!', (23, 24))]
SentencePiece:  [('▁I', (0, 1)), ('▁love', (1, 6)), ('▁machine', (6, 14)), ('▁learning!', (14, 24))]


# 3. Models

```markdown
Tokenizer
├── Normalizer
├── PreTokenizer      ← Whitespace / ByteLevel / Metaspace / BertPreTokenizer
├── Model             ← BPE / WordPiece / Unigram
├── PostProcessor
└── Decoder
```

- Word-level Tokenizer
  - "I love you" -> "I", "love", "you"
  - Out-of-vocabulary -> not used in modern LLMs.
- Character-level Tokenizer
  - "love" -> "l", "o", "v", "e"
  - Large sequence length -> not used in modern LLMs.
- BPE, Byte-pair Encoding
  - Pre-tokenizer: split by some criteria (e.g. whitespace), and tokenization occurs within each split.
  - Character-level BPE: starts with each character -> counts all pairs -> merges the frequent pairs until it reaches the vocab size.
  - Byte-level BPE: starts with each byte to cover unicode characters.
  - SentencePiece BPE: no pre-tokenizer, instead converts ' ' -> '_' and tokenizes the whole sentence.
  - BPE-dropout: intentionally skips some merges.
- WordPiece
  - $Score(A, B) \propto \frac{freq(A, B)}{freq(A)freq(B)}$
  - Greedy longest-match-first tokenization.
- Unigram
  - Starts with large candidates -> measures the explainability -> prunes the weak candidates.
  - Subword regularization: randomly choose the candidate from top K instead of the best one.

In [4]:
corpus = [
    "I love machine learning",
    "I love natural language processing",
    "machine learning is interesting",
    "natural language processing is interesting",
    "learning machines can learn",
]

## Character-level BPE

In [5]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()   # set a pre-tokenizer.

# Train.
trainer = trainers.BpeTrainer(
    vocab_size=50,
    min_frequency=1,
    special_tokens=["<UNK>"],
)
tokenizer.train_from_iterator(corpus, trainer)

# Tokenize.
text = "I love machine killing!"
print(tokenizer.encode(text).tokens)

['I', 'lo', 've', 'machin', 'e', '<UNK>', 'i', 'l', 'l', 'ing', '<UNK>']


## Byte-level BPE

In [6]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
tokenizer.decoder = decoders.ByteLevel()

# Train.
trainer = trainers.BpeTrainer(
    vocab_size=300,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # don't treat alphabets as <UNK>.
    special_tokens=["<UNK>"],
)
tokenizer.train_from_iterator(corpus, trainer)

# Tokenize.
text = "I love machine killing!"
print(tokenizer.encode(text).tokens)
print(tokenizer.decode(tokenizer.encode(text).ids))

['ĠI', 'Ġlov', 'e', 'Ġmachine', 'Ġ', 'k', 'i', 'l', 'l', 'ing', '!']
 I love machine killing!


## SentencePiece BPE

In [7]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁",)
tokenizer.decoder = decoders.Metaspace(replacement="▁",)

# Train.
trainer = trainers.BpeTrainer(
    vocab_size=50,
    min_frequency=1,
    special_tokens=["<UNK>"],
)

# Tokenize.
tokenizer.train_from_iterator(corpus, trainer)
print(tokenizer.encode(text).tokens)

['▁', 'I', '▁l', 'ov', 'e', '▁machin', 'e', '▁', '<UNK>', 'i', 'l', 'l', 'ing', '<UNK>']


## WordPiece

In [8]:
tokenizer = Tokenizer(
    models.WordPiece(unk_token="[UNK]")
)
tokenizer.pre_tokenizer = pre_tokenizers.BertPreTokenizer()

# Train.
trainer = trainers.WordPieceTrainer(
    vocab_size=50,
    min_frequency=1,
    special_tokens=["[UNK]"],
)
tokenizer.train_from_iterator(corpus, trainer)

# Tokenize.
print(tokenizer.encode(text).tokens)

['I', 'lo', '##v', '##e', 'machin', '##e', '[UNK]', '[UNK]']


## Unigram

In [9]:
tokenizer = Tokenizer(models.Unigram())
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁")
tokenizer.decoder = decoders.Metaspace(replacement="▁")

# Train.
trainer = trainers.UnigramTrainer(
    vocab_size=50,
    special_tokens=["<UNK>"],
    unk_token="<UNK>",
)

# Tokenize.
tokenizer.train_from_iterator(corpus, trainer)
print(tokenizer.encode(text).tokens)

['▁', 'I', '▁l', 'o', 'v', 'e', '▁machine', '▁', 'k', 'i', 'l', 'l', 'in', 'g', '!']


# 4. Post-processor

In [10]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors

# 1. Tokenizer
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# 2. Train
texts = [
    "I love you.",
    "I hate you.",
    "You love me."
]
trainer = trainers.BpeTrainer(
    vocab_size=100,
    special_tokens=[
        "[UNK]",
        "[CLS]",
        "[SEP]",
        "[PAD]"
    ]
)
tokenizer.train_from_iterator(texts, trainer=trainer)

# 3. Get special-token IDs
cls_id = tokenizer.token_to_id("[CLS]")
sep_id = tokenizer.token_to_id("[SEP]")

# 4. Post-processor
post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B [SEP]",
    special_tokens=[
        ("[CLS]", cls_id),
        ("[SEP]", sep_id),
    ],
)
tokenizer.post_processor = post_processor

# 5. Test
encoded = tokenizer.encode(
    "I love you",
    "Do you love me?",
)

print(encoded.ids)
print(encoded.tokens)

[1, 5, 21, 20, 2, 0, 12, 20, 21, 25, 0, 2]
['[CLS]', 'I', 'love', 'you', '[SEP]', '[UNK]', 'o', 'you', 'love', 'me', '[UNK]', '[SEP]']
